# Silver: Fato Vendas

**Objetivo:** transformar a tabela Bronze de Vendas na fato definitiva da Silver, aplicando:
1. Casting de tipos (valor unitário: texto → decimal)
2. Cálculo de `valor_total_moeda_local` (quantidade × valor unitário)
3. Padronização de texto (`loja_cidade`), para compatibilizar com as chaves já padronizadas em `silver.lojas`
4. Enriquecimento via JOIN com `silver.lojas` e `silver.representantes`, usando **join efetivo por data** (considerando qual versão SCD2 estava vigente na data da venda)
5. Enriquecimento via JOIN com `silver.dim_cambio`, calculando `cambio_usado` e `valor_total_usd`

**Fonte:** tabela Delta `bronze.vendas`.

**Destino:** tabela Delta `silver.fato_vendas`.

**Modo de execução:** streaming incremental, com **append simples** (vendas são transações imutáveis, não sofrem SCD2).

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import col, regexp_replace, upper, translate, current_timestamp
from pyspark.sql.functions import round as spark_round

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.silver")

print("Schema 'silver' verificado/criado com sucesso.")

In [0]:
# Leitura, casting e padronização inicial

caracteres_com_acento = "áàãâäéèêëíìîïóòõôöúùûüçñÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
caracteres_sem_acento = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

df_stream_bronze_vendas = spark.readStream.format("delta").table("poc_latam_food.bronze.vendas")

df_vendas_tratado = (
    df_stream_bronze_vendas
    .withColumn(
        "valor_unitario_moeda_local",
        regexp_replace(col("valor_unitario_moeda_local"), ",", ".").cast("decimal(10,2)")
    )
    .withColumn("valor_total_moeda_local", col("quantidade") * col("valor_unitario_moeda_local"))
    .withColumn("loja_cidade", upper(translate(col("loja_cidade"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("pais", upper(translate(col("pais"), caracteres_com_acento, caracteres_sem_acento)))
)

print("Streaming de vendas configurado, com casting e padronização aplicados.")

In [0]:
# Leitura das dimensões para o JOIN

df_lojas_silver = spark.table("poc_latam_food.silver.lojas")
df_representantes_silver = spark.table("poc_latam_food.silver.representantes")
df_cambio_silver = spark.table("poc_latam_food.silver.dim_cambio")

print(f"Lojas: {df_lojas_silver.count()} | Representantes: {df_representantes_silver.count()} | Câmbio: {df_cambio_silver.count()}")

In [0]:
# JOIN efetivo por data - Lojas

from pyspark.sql.functions import lit

df_vendas_com_loja = (
    df_vendas_tratado.alias("v")
    .join(
        df_lojas_silver.alias("l"),
        (col("v.loja_cidade") == col("l.nome_cidade")) &
        (col("v.data_venda").cast("date") >= col("l.data_inicio")) &
        (
            col("l.data_fim").isNull() |
            (col("v.data_venda").cast("date") < col("l.data_fim"))
        ),
        "left"
    )
    .select(
        "v.*",
        col("l.loja_id").alias("loja_id")
    )
)

print("JOIN com Lojas configurado.")

In [0]:
# JOIN efetivo por data - Representantes

df_vendas_com_representante = (
    df_vendas_com_loja.alias("v")
    .join(
        df_representantes_silver.alias("r"),
        (col("v.representante_id") == col("r.representante_id")) &
        (col("v.data_venda").cast("date") >= col("r.data_inicio")) &
        (
            col("r.data_fim").isNull() |
            (col("v.data_venda").cast("date") < col("r.data_fim"))
        ),
        "left"
    )
    .select(
        "v.*",
        col("r.nome").alias("representante_nome")
    )
)

print("JOIN com Representantes configurado.")

In [0]:
# JOIN com Câmbio - cálculo do valor em USD

df_vendas_com_cambio = (
    df_vendas_com_representante.alias("v")
    .join(
        df_cambio_silver.alias("c"),
        (col("v.moeda") == col("c.moeda")) &
        (col("v.data_venda").cast("date") == col("c.data_cotacao")),
        "left"
    )
    .select(
        "v.*",
        col("c.taxa_para_usd").alias("cambio_usado"),
        spark_round(col("v.valor_total_moeda_local") / col("c.taxa_para_usd"), 2).alias("valor_total_usd")
    )
)

print("JOIN com Câmbio configurado.")

In [0]:
# Seleção final e gravação (streaming, append)

df_fato_vendas_final = (
    df_vendas_com_cambio
    .select(
        "venda_id",
        col("data_venda").cast("date").alias("data_venda"),
        "produto_id",
        "loja_id",
        "loja_cidade",
        "representante_id",
        "representante_nome",
        "pais",
        "quantidade",
        "valor_unitario_moeda_local",
        "valor_total_moeda_local",
        "moeda",
        "cambio_usado",
        "valor_total_usd",
        "data_ingestao",
        "data_ingestao_bronze",
        current_timestamp().alias("data_ingestao_silver")
    )
)

caminho_checkpoint_silver_vendas = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/silver/fato_vendas"

(
    df_fato_vendas_final
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_silver_vendas)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("poc_latam_food.silver.fato_vendas")
)

print("Silver de Fato Vendas concluído.")

In [0]:
# Validação visual - silver.fato_vendas

spark.table("poc_latam_food.silver.fato_vendas").display()

print(f"Total de linhas: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

print("Vendas sem cotação de câmbio (esperado para dados anteriores a hoje):")
spark.table("poc_latam_food.silver.fato_vendas").filter("valor_total_usd IS NULL").groupBy("data_venda").count().display()

print("Vendas COM cotação de câmbio (esperado apenas para hoje):")
spark.table("poc_latam_food.silver.fato_vendas").filter("valor_total_usd IS NOT NULL").groupBy("data_venda").count().display()